# 🏔️ Python Heap / Priority Queue — The Master Guide
### *From Zero to Interview-Ready*

---

**Mental Model:** A heap is a **tournament bracket** running upside down. The winner (smallest or largest) is always at the top — root. Add a new player: it bubbles up if it beats its parent. Remove the winner: the last player fills the slot and sinks down until correct. Every add/remove costs O(log n) — only the path up or down the bracket, not the whole crowd.

---

## Table of Contents

1. [Visual Model — How a Heap Works](#1)
2. [Creating / Setup](#2)
3. [API Quick Reference](#3)
4. [Decision Map — When to Use What](#4)
5. [Pattern 1 — Kth Largest Element (LC 215)](#5)
6. [Pattern 2 — Top K Frequent Elements (LC 347)](#6)
7. [Pattern 3 — Merge K Sorted Lists (LC 23)](#7)
8. [Pattern 4 — Find Median from Data Stream (LC 295)](#8)
9. [Pattern 5 — K Closest Points to Origin (LC 973)](#9)
10. [Full Decision Map](#10)
11. [Cheat Sheet + Templates](#11)

<a id='1'></a>

## 1. 🗺️ Visual Model — How a Heap Works

---

### Min-Heap as a Binary Tree

```
             1          ← root (minimum) — always here, always O(1) peek
           /   \
          3     2       ← children are ALWAYS >= parent
         / \   / \
        7   4 5   6     ← no ordering between siblings — just parent vs child
```

**Rule:** Every parent ≤ both children. That is the ONLY rule. Siblings can be in any order.

---

### Same Heap as an Array

```
Index:  0   1   2   3   4   5   6
Array: [1,  3,  2,  7,  4,  5,  6]
        ↑
       root
```

**Index math — no pointers needed:**
```
parent(i)      = (i - 1) // 2
left_child(i)  = 2*i + 1
right_child(i) = 2*i + 2

Example — node at index 1 (value=3):
  parent      = (1-1)//2 = 0  → value 1  ✓ (1 ≤ 3)
  left_child  = 2*1+1   = 3  → value 7  ✓ (3 ≤ 7)
  right_child = 2*1+2   = 4  → value 4  ✓ (3 ≤ 4)
```

---

### Heapify Direction

```
PUSH (add new element):
  Place at end → bubble UP → swap with parent while smaller
  Cost: O(log n) — only the path from leaf to root

POP (remove minimum):
  Grab root → move last element to root → sink DOWN → swap with smaller child while larger
  Cost: O(log n) — only the path from root to leaf

PEEK:
  h[0] — the root. Always there. No movement needed.
  Cost: O(1)
```

---

### Why Does This Matter?

```
You need the k-th largest element from 1,000,000 numbers.

  Sort everything:        O(n log n)  — 1,000,000 × 20 = 20,000,000 ops
  Heap of size k:         O(n log k)  — 1,000,000 × log(k) ops

  If k=10:  heap does 1,000,000 × 3.3 = 3,300,000 ops
  That is 6× faster. The heap gives you the best element in O(1)
  WITHOUT sorting everything.
```

<a id='2'></a>

## 2. 🔧 Creating / Setup

In [ ]:
import heapq

# --- Empty heap ---
h = []                          # just a plain list — heapq operates on lists
print("Empty heap:", h)

# --- heapify: convert existing list to heap IN PLACE, O(n) ---
data = [5, 3, 8, 1, 9, 2, 7]
heapq.heapify(data)             # rearranges data; does NOT sort it
print("After heapify:", data)   # smallest is at index 0; siblings unordered

# --- heappush: add element, maintain heap property ---
heapq.heappush(data, 0)
print("After push(0):", data)   # 0 bubbled up to root

# --- heappop: remove and return the SMALLEST element ---
smallest = heapq.heappop(data)
print("Popped:", smallest)      # 0
print("After pop:", data)

# --- heappushpop: push then pop in ONE call — faster than two calls ---
result = heapq.heappushpop(data, 100)  # pushes 100, immediately pops smallest
print("heappushpop(100) returned:", result)

# --- nlargest: top k largest — no full sort needed ---
top3 = heapq.nlargest(3, data)
print("Top 3 largest:", top3)

# --- nsmallest: bottom k smallest ---
bot3 = heapq.nsmallest(3, data)
print("Bottom 3 smallest:", bot3)

# --- MAX-HEAP TRICK: Python only has min-heap ---
# Negate values on push; negate again on pop to recover original
max_heap = []
for val in [5, 3, 8, 1, 9]:
    heapq.heappush(max_heap, -val)   # store negative

max_val = -heapq.heappop(max_heap)   # negate again = original max
print("Max-heap popped max:", max_val)  # 9
print("Max-heap state (negated):", max_heap)

<a id='3'></a>

## 3. 📋 API Quick Reference

```
OPERATION                 COMPLEXITY   WHAT IT DOES
─────────────────────────────────────────────────────────────────────
heapq.heappush(h, x)      O(log n)     Add x, maintain heap property
heapq.heappop(h)          O(log n)     Remove and return smallest
h[0]                      O(1)         Peek at minimum — no removal
heapq.heapify(list)       O(n)         Convert list to heap in-place
heapq.heappushpop(h, x)   O(log n)     Push x then pop min — faster than separate calls
heapq.heapreplace(h, x)   O(log n)     Pop min then push x — raises if empty
heapq.nlargest(k, iter)   O(n log k)   Top k largest — better than sort for small k
heapq.nsmallest(k, iter)  O(n log k)   Bottom k smallest
len(h)                    O(1)         Number of elements
─────────────────────────────────────────────────────────────────────

MAX-HEAP TRICK:  negate values on push, negate again on pop
  heappush(h, -val)  →  heappop(h) returns -max_val  →  -(-max_val) = max_val

THINGS YOU DO NOT DO:
❌  h.pop()       — use heapq.heappop(h), not list.pop()
❌  h[0] = x      — direct assignment breaks heap property
❌  Sorting the whole array just to get the k-th element
```

In [ ]:
import heapq

# --- Live demo: min-heap operations step by step ---
h = []
print("=== MIN-HEAP DEMO ===")

for val in [4, 1, 7, 3, 9, 2]:
    heapq.heappush(h, val)
    print(f"  push({val}) → heap: {h}  | peek (min): {h[0]}")

print()
print(f"  pop → {heapq.heappop(h)}  | heap: {h}")
print(f"  pop → {heapq.heappop(h)}  | heap: {h}")

print()
print("=== MAX-HEAP DEMO (negate trick) ===")
max_h = []
for val in [4, 1, 7, 3, 9, 2]:
    heapq.heappush(max_h, -val)          # store negated
    print(f"  push({val}) stored as {-val} → heap: {max_h}  | peek (max): {-max_h[0]}")

print()
print(f"  pop max → {-heapq.heappop(max_h)}  | heap: {max_h}")

print()
print("=== HEAPIFY DEMO ===")
raw = [8, 3, 5, 1, 9, 2, 7]
print(f"  Before heapify: {raw}")
heapq.heapify(raw)                       # in-place, O(n)
print(f"  After  heapify: {raw}  | min at [0]: {raw[0]}")

<a id='4'></a>

## 4. 🧭 Decision Map — When to Use What

```
SIGNAL IN THE PROBLEM                   WHAT TO DO
──────────────────────────────────────────────────────────────────────
"kth largest / kth smallest"            min-heap of size k
"top k most frequent"                   Counter + nlargest
"stream of numbers, find median"        two heaps: max-heap left, min-heap right
"merge k sorted lists/arrays"           heap with (val, list_idx, elem_idx)
"always need the minimum/maximum"       heap (not sorted list)
"k closest points"                      max-heap of size k (negate distance)
"continuous stream, priority ops"       heap beats sorted list re-insertion O(n)
──────────────────────────────────────────────────────────────────────
```

<a id='5'></a>

## 5. 🎯 Pattern 1 — Kth Largest Element (LC 215)

---

**PROBLEM:** Given an unsorted array, find the kth largest element. No full sort allowed.

---

**TRICK:** Maintain a **min-heap of size k**. The root is always the kth largest.

```
Why min-heap for "largest"?
  You keep the k biggest numbers seen so far.
  The smallest of those k numbers = the kth largest overall.
  A min-heap gives you that smallest one at the root — instant O(1) peek.
  When a new number arrives that beats the root, the old kth-largest is evicted.
```

---

**SLOW MOTION TRACE:** `nums=[3,2,1,5,6,4]`, `k=2`

```
We want the 2nd largest. Answer will be 5.

Process 3:  heap=[3]           size < k, just push
Process 2:  heap=[2,3]         size < k, just push
Process 1:  heap=[2,3]  1≤root(2), skip — 1 can't be in top 2
Process 5:  heap=[3,5]  5>root(2), push 5, pop 2 — heap=[3,5]
Process 6:  heap=[5,6]  6>root(3), push 6, pop 3 — heap=[5,6]
Process 4:  heap=[5,6]  4≤root(5), skip — 4 can't beat 5th position

Root of heap = 5 = 2nd largest ✓
```

---

**KEY INSIGHT:** The heap root is the kth largest because exactly k-1 elements in the heap are larger than it.

---

```
TIME:  O(n log k)  — n elements processed, each heap op is O(log k)
SPACE: O(k)        — heap never exceeds k elements
```

In [ ]:
import heapq
from typing import List

def find_kth_largest(nums: List[int], k: int) -> int:
    """
    LC 215 — Kth Largest Element in an Array
    Approach: maintain a min-heap of size k.
    Time:  O(n log k) — n pushes, each O(log k)
    Space: O(k) — heap never exceeds k elements
    """
    heap = []                               # min-heap of size k

    for num in nums:
        heapq.heappush(heap, num)           # always push first
        if len(heap) > k:                   # heap grew beyond k
            heapq.heappop(heap)             # evict the smallest — it can't be kth largest

    # heap[0] = smallest of the top-k = the kth largest overall
    return heap[0]


# --- SLOW MOTION TRACE (matches markdown above) ---
def find_kth_largest_trace(nums: List[int], k: int) -> int:
    """Same algorithm with debug prints."""
    heap = []
    for num in nums:
        heapq.heappush(heap, num)
        if len(heap) > k:
            evicted = heapq.heappop(heap)
            print(f"  push({num}) → evict({evicted}) → heap={heap}")
        else:
            print(f"  push({num}) → heap={heap}  (filling up)")
    print(f"  Result: heap[0] = {heap[0]}")
    return heap[0]


# --- TEST HARNESS ---
def test_harness(fn, test_cases):
    passed = 0
    for i, (args, expected) in enumerate(test_cases):
        result = fn(*args)
        status = "PASS" if result == expected else "FAIL"
        if status == "FAIL":
            print(f"  [{status}] Test {i+1}: args={args} → got {result}, expected {expected}")
        passed += (1 if status == "PASS" else 0)
    print(f"  {passed}/{len(test_cases)} tests passed.")


tests = [
    (([3,2,1,5,6,4], 2), 5),
    (([3,2,3,1,2,4,5,5,6], 4), 4),
    (([1], 1), 1),
    (([7,6,5,4,3,2,1], 3), 5),
    (([1,2], 1), 2),
]

print("=== TRACE ===")
find_kth_largest_trace([3,2,1,5,6,4], 2)

print("\n=== TEST HARNESS ===")
test_harness(find_kth_largest, tests)

print("find_kth_largest defined.")

<a id='6'></a>

## 6. 📊 Pattern 2 — Top K Frequent Elements (LC 347)

---

**PROBLEM:** Given an array, return the k most frequent elements.

---

**TRICK:** Count frequencies with `Counter`, then use `heapq.nlargest` to grab the top k by count.

```
Two-step pipeline:
  1. Counter([1,1,1,2,2,3]) → {1:3, 2:2, 3:1}
  2. nlargest(2, counter, key=count) → [1, 2]
```

---

**SLOW MOTION TRACE:** `nums=[1,1,1,2,2,3]`, `k=2`

```
Step 1 — Count:
  {1: 3, 2: 2, 3: 1}

Step 2 — Build min-heap of (count, element) pairs, size k:
  Process (3, 1): heap=[(3,1)]         size < k
  Process (2, 2): heap=[(2,2),(3,1)]   size == k
  Process (1, 3): 1 < heap root (2), skip — 3 is not frequent enough

  Result: [1, 2] ✓
```

---

**KEY INSIGHT:** `nlargest(k, counter.keys(), key=counter.get)` does the heap work for you. O(n log k), not O(n log n).

---

```
TIME:  O(n log k)  — counting is O(n); nlargest heap is O(n log k)
SPACE: O(n)        — Counter stores at most n unique elements
```

In [ ]:
import heapq
from collections import Counter
from typing import List

def top_k_frequent(nums: List[int], k: int) -> List[int]:
    """
    LC 347 — Top K Frequent Elements
    Approach: Counter for frequency, nlargest for top k.
    Time:  O(n log k) — Counter O(n), nlargest O(n log k)
    Space: O(n) — Counter dictionary
    """
    count = Counter(nums)                               # {element: frequency}
    # nlargest picks by key=count.get → sorts by frequency, not value
    return heapq.nlargest(k, count.keys(), key=count.get)


# --- TRACE VERSION ---
def top_k_frequent_trace(nums: List[int], k: int) -> List[int]:
    """Same with debug prints."""
    count = Counter(nums)
    print(f"  Frequencies: {dict(count)}")
    result = heapq.nlargest(k, count.keys(), key=count.get)
    print(f"  Top {k} frequent: {result}")
    return result


# --- TEST HARNESS ---
def test_harness(fn, test_cases):
    passed = 0
    for i, (args, expected) in enumerate(test_cases):
        result = sorted(fn(*args))              # sort both — order doesn't matter
        exp_sorted = sorted(expected)
        status = "PASS" if result == exp_sorted else "FAIL"
        if status == "FAIL":
            print(f"  [{status}] Test {i+1}: args={args} → got {result}, expected {exp_sorted}")
        passed += (1 if status == "PASS" else 0)
    print(f"  {passed}/{len(test_cases)} tests passed.")


tests = [
    (([1,1,1,2,2,3], 2), [1,2]),
    (([1], 1), [1]),
    (([4,4,4,3,3,2,1], 2), [4,3]),
    (([1,2,3,4,5], 3), [1,2,3]),      # all equal frequency — any 3 valid
]

print("=== TRACE ===")
top_k_frequent_trace([1,1,1,2,2,3], 2)

print("\n=== TEST HARNESS ===")
test_harness(top_k_frequent, tests[:3])  # skip all-equal-freq test

print("top_k_frequent defined.")

<a id='7'></a>

## 7. 🔗 Pattern 3 — Merge K Sorted Lists (LC 23)

---

**PROBLEM:** Given k sorted lists, merge them into one sorted list efficiently.

---

**TRICK:** Use a min-heap storing **(value, list_index, element_index)** tuples. Always pull the global minimum next.

```
Why a tuple?
  (val, list_idx, elem_idx) lets the heap compare by val first.
  list_idx breaks ties — prevents comparing lists directly (which would error).
  elem_idx tracks where we are in each list.
```

---

**SLOW MOTION TRACE:** `lists=[[1,4,5],[1,3,4],[2,6]]`

```
Initial heap — first element from each list:
  (1, 0, 0)  → value=1, from list 0, index 0
  (1, 1, 0)  → value=1, from list 1, index 0
  (2, 2, 0)  → value=2, from list 2, index 0

Pop (1, 0, 0) → output: [1]  | push next from list 0: (4, 0, 1)
  heap: [(1,1,0), (4,0,1), (2,2,0)]

Pop (1, 1, 0) → output: [1,1]  | push next from list 1: (3, 1, 1)
  heap: [(2,2,0), (4,0,1), (3,1,1)]

Pop (2, 2, 0) → output: [1,1,2]  | push next from list 2: (6, 2, 1)
  heap: [(3,1,1), (4,0,1), (6,2,1)]

Pop (3, 1, 1) → output: [1,1,2,3]  | push (4, 1, 2)
Pop (4, 0, 1) → output: [1,1,2,3,4]  | push (5, 0, 2)
Pop (4, 1, 2) → output: [1,1,2,3,4,4]  | list 1 exhausted
Pop (5, 0, 2) → output: [1,1,2,3,4,4,5]  | list 0 exhausted
Pop (6, 2, 1) → output: [1,1,2,3,4,4,5,6]  | list 2 exhausted

Done ✓
```

---

**KEY INSIGHT:** Heap size stays at most k throughout. Each pop is the global minimum — no re-scanning all lists.

---

```
TIME:  O(N log k)  — N = total elements, each heap op is O(log k)
SPACE: O(k)        — heap holds one element per list at most
```

In [ ]:
import heapq
from typing import List

def merge_k_sorted(lists: List[List[int]]) -> List[int]:
    """
    LC 23 — Merge K Sorted Lists (array version for clarity)
    Approach: min-heap of (value, list_idx, elem_idx) tuples.
    Time:  O(N log k) — N total elements, each heap op O(log k)
    Space: O(k) — heap holds at most one element per list
    """
    heap = []
    result = []

    # Seed heap with first element of each list
    for li, lst in enumerate(lists):
        if lst:                                     # skip empty lists
            heapq.heappush(heap, (lst[0], li, 0))  # (val, list_idx, elem_idx)

    while heap:
        val, li, ei = heapq.heappop(heap)   # pull global minimum
        result.append(val)                   # add to output

        next_ei = ei + 1                     # advance pointer in that list
        if next_ei < len(lists[li]):         # if that list has more elements
            next_val = lists[li][next_ei]
            heapq.heappush(heap, (next_val, li, next_ei))

    return result


# --- TRACE VERSION ---
def merge_k_sorted_trace(lists: List[List[int]]) -> List[int]:
    """Same with debug prints."""
    heap = []
    result = []
    for li, lst in enumerate(lists):
        if lst:
            heapq.heappush(heap, (lst[0], li, 0))
    print(f"  Initial heap: {heap}")
    while heap:
        val, li, ei = heapq.heappop(heap)
        result.append(val)
        next_ei = ei + 1
        if next_ei < len(lists[li]):
            next_val = lists[li][next_ei]
            heapq.heappush(heap, (next_val, li, next_ei))
        print(f"  pop({val}) → result={result}  heap={heap}")
    return result


# --- TEST HARNESS ---
def test_harness(fn, test_cases):
    passed = 0
    for i, (args, expected) in enumerate(test_cases):
        result = fn(*args)
        status = "PASS" if result == expected else "FAIL"
        if status == "FAIL":
            print(f"  [{status}] Test {i+1}: got {result}, expected {expected}")
        passed += (1 if status == "PASS" else 0)
    print(f"  {passed}/{len(test_cases)} tests passed.")


tests = [
    (([[1,4,5],[1,3,4],[2,6]],), [1,1,2,3,4,4,5,6]),
    (([[],],), []),
    (([[1],[2],[3]],), [1,2,3]),
    (([[1,2,3],[4,5,6]],), [1,2,3,4,5,6]),
    (([[5],[3,4],[1,2]],), [1,2,3,4,5]),
]

print("=== TRACE ===")
merge_k_sorted_trace([[1,4,5],[1,3,4],[2,6]])

print("\n=== TEST HARNESS ===")
test_harness(merge_k_sorted, tests)

print("merge_k_sorted defined.")

<a id='8'></a>

## 8. 📡 Pattern 4 — Find Median from Data Stream (LC 295)

---

**PROBLEM:** Numbers arrive one at a time in a stream. After each arrival, return the current median.

---

**TRICK:** Split the stream into two halves:
```
  LEFT half  → max-heap (negated)   stores the smaller half
  RIGHT half → min-heap             stores the larger half

  Invariants:
    1. All left values ≤ all right values
    2. len(left) == len(right)  OR  len(left) == len(right) + 1
       (left holds the extra element when count is odd)

  Median:
    Even count  → average of left top and right top
    Odd count   → left top (left always holds the extra)
```

---

**SLOW MOTION TRACE:** Adding 1, 2, 3, 4, 5

```
Add 1:
  Push to left (max-heap): left=[-1] right=[]
  left bigger by 1 → OK (left holds extra)
  Median = -left[0] = 1

Add 2:
  Push to left: left=[-2,-1] → balance: move top of left to right
  left=[-1] right=[2]
  Sizes equal → median = (-left[0] + right[0]) / 2 = (1+2)/2 = 1.5

Add 3:
  3 > -left[0]=1 → push to right: right=[2,3]
  right bigger → move top of right to left: right=[3] left=[-2,-1]
  left bigger by 1 → OK
  Median = -left[0] = 2

Add 4:
  4 > -left[0]=2 → push to right: right=[3,4]
  Sizes equal → median = (2+3)/2 = 2.5

Add 5:
  5 > -left[0]=2 → push to right: right=[3,4,5]
  right bigger → move 3 to left: left=[-3,-2,-1] right=[4,5]
  left bigger by 1 → OK
  Median = -left[0] = 3
```

---

**KEY INSIGHT:** Both heap tops are always the two middle values. Median is always a constant-time read.

---

```
TIME:  O(log n) per add_num — two heap ops at most
       O(1)    per find_median — just read two tops
SPACE: O(n) — both heaps combined store all elements
```

In [ ]:
import heapq

class MedianFinder:
    """
    LC 295 — Find Median from Data Stream
    Approach: two heaps — max-heap for lower half, min-heap for upper half.
    Time:  O(log n) per add_num, O(1) per find_median
    Space: O(n)
    """

    def __init__(self):
        self.left = []      # max-heap (negated) — lower half
        self.right = []     # min-heap — upper half

    def add_num(self, num: int) -> None:
        # Step 1: always push to left first
        heapq.heappush(self.left, -num)

        # Step 2: enforce left top ≤ right top (cross-over check)
        if self.right and -self.left[0] > self.right[0]:
            val = -heapq.heappop(self.left)     # move left's max to right
            heapq.heappush(self.right, val)

        # Step 3: balance sizes — left can have at most 1 more than right
        if len(self.left) > len(self.right) + 1:
            val = -heapq.heappop(self.left)     # left too big, move to right
            heapq.heappush(self.right, val)
        elif len(self.right) > len(self.left):
            val = heapq.heappop(self.right)      # right too big, move to left
            heapq.heappush(self.left, -val)

    def find_median(self) -> float:
        if len(self.left) > len(self.right):
            return float(-self.left[0])           # odd count — left holds extra
        return (-self.left[0] + self.right[0]) / 2.0  # even count — average both tops


# --- TEST HARNESS ---
def test_harness(operations, values, expected_medians):
    mf = MedianFinder()
    passed = 0
    total = 0
    for op, val, exp in zip(operations, values, expected_medians):
        if op == "add":
            mf.add_num(val)
            median = mf.find_median()
            status = "PASS" if median == exp else "FAIL"
            print(f"  add({val:2d}) → left={[-x for x in sorted(mf.left, reverse=True)]}  "
                  f"right={sorted(mf.right)}  median={median}  [{status}]")
            passed += (1 if status == "PASS" else 0)
            total += 1
    print(f"  {passed}/{total} tests passed.")


# Test: add 1,2,3,4,5 — matches the markdown trace
ops  = ["add"] * 5
vals = [1, 2, 3, 4, 5]
exps = [1.0, 1.5, 2.0, 2.5, 3.0]

print("=== TRACE (matches markdown) ===")
test_harness(ops, vals, exps)

# Additional test
print("\n=== ADDITIONAL TEST ===")
mf2 = MedianFinder()
for x in [6, 3, 9, 1, 7]:
    mf2.add_num(x)
print(f"  After adding [6,3,9,1,7]: median = {mf2.find_median()}")  # sorted: 1,3,6,7,9 → median=6

print("MedianFinder defined.")

<a id='9'></a>

## 9. 📍 Pattern 5 — K Closest Points to Origin (LC 973)

---

**PROBLEM:** Given a list of 2D points, return the k closest to the origin (0, 0).

---

**TRICK:** Use a **max-heap of size k** on distance (negated). Evict the farthest point when heap exceeds k.

```
Why max-heap?
  We want to keep the k CLOSEST points.
  A max-heap lets us instantly see the FARTHEST of our current candidates.
  When a closer point arrives, we evict the farthest.
  At the end, the heap holds exactly the k closest.

Distance trick:
  sqrt is monotone — skip it. Compare dist² = x²+y².
  Negate dist² for max-heap behavior.
```

---

**SLOW MOTION TRACE:** `points=[[1,3],[-2,2],[5,8],[0,1]]`, `k=2`

```
Distances squared:
  [1,3]  → 1+9   = 10
  [-2,2] → 4+4   = 8
  [5,8]  → 25+64 = 89
  [0,1]  → 0+1   = 1

Process [1,3]  d²=10: heap=[(-10,[1,3])]             size<k
Process [-2,2] d²=8:  heap=[(-10,[1,3]),(-8,[-2,2])] size==k
Process [5,8]  d²=89: 89>10 (heap root=-10)? YES → skip (89 is farther than both)
Process [0,1]  d²=1:  1<10 (heap root=-10)? YES → evict [1,3], add [0,1]
  heap=[(-8,[-2,2]),(-1,[0,1])]

Result: [[-2,2],[0,1]] ✓
```

---

**KEY INSIGHT:** Max-heap of size k maintains the k closest without sorting all n points.

---

```
TIME:  O(n log k)  — n points processed, each heap op O(log k)
SPACE: O(k)        — heap never exceeds k elements
```

In [ ]:
import heapq
from typing import List

def k_closest(points: List[List[int]], k: int) -> List[List[int]]:
    """
    LC 973 — K Closest Points to Origin
    Approach: max-heap of size k on squared distance (negated).
    Time:  O(n log k) — n points, each heap op O(log k)
    Space: O(k) — heap stores at most k points
    """
    heap = []   # max-heap: store (-dist_sq, point)

    for x, y in points:
        dist_sq = x*x + y*y         # no sqrt needed — dist² preserves order
        heapq.heappush(heap, (-dist_sq, [x, y]))  # negate for max-heap

        if len(heap) > k:
            heapq.heappop(heap)     # evict the FARTHEST point (largest dist_sq)

    return [point for _, point in heap]   # extract points, discard distances


# --- TRACE VERSION ---
def k_closest_trace(points: List[List[int]], k: int) -> List[List[int]]:
    """Same with debug prints."""
    heap = []
    for x, y in points:
        dist_sq = x*x + y*y
        heapq.heappush(heap, (-dist_sq, [x, y]))
        if len(heap) > k:
            evicted = heapq.heappop(heap)
            print(f"  push([{x},{y}]) d²={dist_sq} → evict d²={-evicted[0]} {evicted[1]}  heap_size={len(heap)}")
        else:
            print(f"  push([{x},{y}]) d²={dist_sq} → heap_size={len(heap)}")
    result = [point for _, point in heap]
    print(f"  Result: {result}")
    return result


# --- TEST HARNESS ---
def test_harness(fn, test_cases):
    passed = 0
    for i, (args, expected) in enumerate(test_cases):
        result = sorted(fn(*args))          # sort both for order-independent compare
        exp_sorted = sorted(expected)
        status = "PASS" if result == exp_sorted else "FAIL"
        if status == "FAIL":
            print(f"  [{status}] Test {i+1}: got {result}, expected {exp_sorted}")
        passed += (1 if status == "PASS" else 0)
    print(f"  {passed}/{len(test_cases)} tests passed.")


tests = [
    (([[1,3],[-2,2]], 1), [[-2,2]]),                     # sqrt(8) < sqrt(10)
    (([[3,3],[5,-1],[-2,4]], 2), [[3,3],[-2,4]]),        # d²=18,26,20 → top 2: 18,20
    (([[0,1],[1,0]], 2), [[0,1],[1,0]]),                  # both d²=1, keep both
    (([[1,3],[-2,2],[5,8],[0,1]], 2), [[-2,2],[0,1]]),   # matches trace
]

print("=== TRACE ===")
k_closest_trace([[1,3],[-2,2],[5,8],[0,1]], 2)

print("\n=== TEST HARNESS ===")
test_harness(k_closest, tests)

print("k_closest defined.")

<a id='10'></a>

## 10. 🗺️ Full Decision Map

---

```
PATTERN                          SIGNAL                              HEAP TYPE        SIZE    ROOT IS
────────────────────────────────────────────────────────────────────────────────────────────────────────────
Kth Largest (LC 215)             "kth largest"                       min-heap         k       kth largest
Kth Smallest                     "kth smallest"                      max-heap         k       kth smallest
Top K Frequent (LC 347)          "most frequent", "k elements"       Counter+nlargest k       most frequent
Merge K Sorted (LC 23)           "merge", "k sorted lists"           min-heap         k       global min
Median from Stream (LC 295)      "median", "stream", "data stream"   two heaps        n/2     two midpoints
K Closest Points (LC 973)        "closest", "distance", "k points"   max-heap         k       farthest keeper
────────────────────────────────────────────────────────────────────────────────────────────────────────────
```

---

### General Rules

```
"keep the k LARGEST"  → min-heap of size k  (root = smallest of the large = kth largest)
"keep the k SMALLEST" → max-heap of size k  (root = largest of the small = kth smallest)
"need both halves"    → two heaps           (left max-heap, right min-heap)
"merge k streams"     → min-heap of k heads (tuple: val + pointer info)
```

---

### Complexity Summary

```
PATTERN              TIME          SPACE
─────────────────────────────────────────
Kth Largest          O(n log k)    O(k)
Top K Frequent       O(n log k)    O(n)
Merge K Sorted       O(N log k)    O(k)
Median Stream        O(log n)/add  O(n)
K Closest            O(n log k)    O(k)
─────────────────────────────────────────
```

<a id='11'></a>

## 11. 📄 Cheat Sheet + Templates

---

### When to Use a Heap

```
USE A HEAP WHEN:
  ✓ You need the k-th largest/smallest repeatedly
  ✓ Elements arrive in a stream (you cannot sort ahead of time)
  ✓ You need continuous access to min or max without full re-sort
  ✓ Merging multiple sorted sources
  ✓ You need two balanced halves (median problem)

DO NOT USE A HEAP WHEN:
  ✗ You need random access by index — use an array
  ✗ You need the full sorted order — just sort
  ✗ k == n — sort is simpler and same complexity O(n log n)
```

---

### O(log n) Operations

```
heappush    — add element
heappop     — remove min
heappushpop — push then pop (one operation, faster)
heapreplace — pop then push (one operation, raises if empty)

O(1):  h[0]  (peek min)
O(n):  heapify (build from scratch)
```

---

### Templates

**Kth-Largest Window:**
```python
import heapq

def kth_largest_window(nums, k):
    heap = []                          # min-heap of size k
    for num in nums:
        heapq.heappush(heap, num)
        if len(heap) > k:
            heapq.heappop(heap)        # evict smallest — not in top k
    return heap[0]                     # root = kth largest
```

**Two-Heap Median:**
```python
import heapq

left, right = [], []    # left=max-heap (negated), right=min-heap

def add(num):
    heapq.heappush(left, -num)                     # always push to left
    if right and -left[0] > right[0]:              # left top > right top?
        heapq.heappush(right, -heapq.heappop(left))  # move left top to right
    if len(left) > len(right) + 1:                 # left too big?
        heapq.heappush(right, -heapq.heappop(left))
    elif len(right) > len(left):                   # right too big?
        heapq.heappush(left, -heapq.heappop(right))

def median():
    if len(left) > len(right):
        return float(-left[0])                     # odd: left holds extra
    return (-left[0] + right[0]) / 2.0             # even: average both tops
```

**K-Way Merge:**
```python
import heapq

def k_way_merge(lists):
    heap = []
    result = []
    for li, lst in enumerate(lists):               # seed with first of each list
        if lst:
            heapq.heappush(heap, (lst[0], li, 0))
    while heap:
        val, li, ei = heapq.heappop(heap)          # global minimum
        result.append(val)
        if ei + 1 < len(lists[li]):                # advance pointer
            heapq.heappush(heap, (lists[li][ei+1], li, ei+1))
    return result
```

---

### Gotchas

```
GOTCHA 1 — Python only has min-heap.
  Fix: negate values. Push -x, pop gives -max → negate again = max.

GOTCHA 2 — Tuples with non-comparable second elements.
  heap.push((dist, [x,y]))  → ERROR if two dists are equal (can't compare lists)
  Fix: add a tiebreaker. heap.push((dist, i, [x,y])) — index i always breaks tie.

GOTCHA 3 — heapify is O(n), not O(n log n).
  Building a heap from scratch is faster than pushing one by one.
  Use heapify when you have all elements upfront.

GOTCHA 4 — heap[0] is O(1), but h.pop() is NOT heappop.
  h.pop() removes the LAST element (list semantics).
  heapq.heappop(h) removes the root (heap semantics). Always use heapq.heappop.

GOTCHA 5 — Modifying heap elements directly breaks heap property.
  Never do h[i] = new_val. Pop and re-push instead.
  For lazy deletion: push (new_val, id) and skip stale entries on pop.
```

## 12. 🏁 Summary

---

```
                    🏔️ HEAP MASTER GUIDE
                    ════════════════════

             HEAP (tournament bracket, upside down)
                         root
                        /    \
                  left half  right half
                  (smaller)  (larger)

  ┌─────────────────────────────────────────────────────────┐
  │  OPERATION        COST      PYTHON CALL                 │
  │  ─────────────────────────────────────────────────────  │
  │  peek min         O(1)      h[0]                        │
  │  push             O(log n)  heapq.heappush(h, x)        │
  │  pop min          O(log n)  heapq.heappop(h)            │
  │  build from list  O(n)      heapq.heapify(lst)          │
  └─────────────────────────────────────────────────────────┘

  FIVE PATTERNS:
  ┌──────────────────────────────────────────────────────────────────┐
  │  1. Kth Largest    → min-heap size k, root = answer             │
  │  2. Top K Frequent → Counter + nlargest                         │
  │  3. Merge K Lists  → min-heap of (val, list_idx, elem_idx)      │
  │  4. Stream Median  → max-heap left + min-heap right             │
  │  5. K Closest      → max-heap size k, negate distance           │
  └──────────────────────────────────────────────────────────────────┘

  THE ONE RULE:
    "Keep k best" → heap of size k.
    Need k LARGEST → min-heap (root = kth largest).
    Need k SMALLEST → max-heap (root = kth smallest).
    Need both halves → two heaps.
    Need global min from k sources → min-heap of k heads.
```

---

*End of Heap / Priority Queue Master Guide — Sean Edition*